# [심화] Model Context Protocol

이번 실습에서는 MCP 서버를 연결할 때 필요한 추가적인 기능들을 학습합니다.   

- 채널 조회 / 메시지 전송 / 리액션 / 스레드 답글을 제공하는 Slack MCP 서버
- 채널 미지정 시 사용자에게 되묻는 Elicitation 흐름 (자동 응답 콜백과 `input()` 인터랙티브 콜백)
- supergateway로 PlayWright MCP(stdio)를 HTTP 엔드포인트로 노출한 뒤 Agent에 연결
- Stateless / Stateful MCP의 세션 수명 관리 차이

## 1. 학습 목표

- MCP Elicitation 메커니즘을 이해하고, 자동 응답 콜백과 사용자 입력 콜백(`input()`)을 둘 다 구현해 봅니다.
- `slack_sdk` 기반 MCP 서버를 구동하고, dry-run 모드와 실제 토큰 모드를 구분해서 다룹니다.
- supergateway로 PlayWright MCP(stdio)를 HTTP 엔드포인트로 노출한 뒤 Agent에 연결합니다.
- Stateless / Stateful MCP의 차이를 파악하고, `async with client.session(...)` 패턴을 사용해 봅니다.

## 2. 환경 설정

실습에 앞서, 다음 과정을 먼저 준비합니다.

### [Slack 사전 설정하기]
Slack App을 생성하고, 토큰과 채널 ID를 `.env`에 등록해야 합니다.

#### Slack 아이디와 Workspace 준비

1. Slack 계정으로 Workspace에 로그인합니다.
계정이 없는 경우 새로 가입합니다.  
https://slack.com/get-started#/createnew  
기존에 사용하고 있는 Workspace를 사용해도 됩니다.  
(관리자 권한이 필요합니다)


#### Slack App 생성

1. https://api.slack.com/apps 에 접속해 Create New App에서 From scratch를 선택합니다.
2. App Name (예: `langchain-mcp-bot`)과 설치할 Workspace를 선택한 뒤 Create App을 누릅니다.

#### Bot Token Scopes 부여

좌측 메뉴 OAuth & Permissions의 Scopes에서 Bot Token Scopes로 들어가, 아래 권한을 모두 추가합니다.
이 노트북의 도구가 호출하는 Slack API에 1:1로 대응됩니다.

| Scope | 사용하는 도구 / API |
|---|---|
| `channels:read` | `get_channel_info` (`conversations.info`) |
| `channels:history` | `get_channel_messages` (`conversations.history`) |
| `chat:write` | `post_message`, `reply_to_thread` (`chat.postMessage`) |
| `reactions:write` | `add_reaction` (`reactions.add`) |
| `users:read` | 메시지 작성자 이름 조회 (`users.info`) |

> 비공개(private) 채널을 쓰려면 채널 관련 스코프를 `groups:read`, `groups:history`로 추가하세요.

#### 워크스페이스에 앱 설치 + Bot Token 발급

같은 화면 상단의 Install to Workspace를 클릭한 뒤 권한을 승인합니다.
설치가 끝나면 Bot User OAuth Token (`xoxb-...`)이 발급됩니다. 이 값이 `SLACK_BOT_TOKEN`입니다.

#### 채널에 봇 초대 + Channel ID 확인

1. Slack 클라이언트에서 사용할 채널을 열고, 메시지 입력창에 `/invite @앱이름`으로 봇을 초대합니다. (초대하지 않으면 `not_in_channel` 오류가 납니다.)
2. 채널 이름을 클릭한 뒤 하단의 Channel ID (예: `C0123456789`)를 복사합니다. 웹에서는 채널 URL 끝부분(`https://app.slack.com/client/.../C0123456789`)에서도 확인할 수 있습니다.

#### 5. `.env`에 키 저장

프로젝트 루트의 `.env` 파일에 두 줄을 추가합니다.

```
SLACK_BOT_TOKEN=xoxb-...
SLACK_CHANNEL_ID=C0123456789
```

> 두 값 중 하나라도 비어 있으면 MCP 서버는 자동으로 dry-run 모드로 시작하고, 모든 도구가 더미 응답을 돌려줍니다 (실습 흐름 확인용).
>
> Scope를 변경했다면 OAuth & Permissions 화면 상단의 Reinstall to Workspace를 눌러야 새 토큰에 반영됩니다.



필요한 환경변수:
- `OPENAI_API_KEY`
- `SLACK_BOT_TOKEN`, `SLACK_CHANNEL_ID` (없으면 `slack_mcp_server.py`가 자동 dry-run으로 시작합니다)

In [ ]:
%pip install langchain langchain-openai fastmcp langchain-mcp-adapters slack-sdk slack-bolt httpx -q

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv('.env', override=True)

if os.environ.get('OPENAI_API_KEY'):
    print('OpenAI API 키 설정 완료!')
if os.environ.get('SLACK_BOT_TOKEN') and os.environ.get('SLACK_CHANNEL_ID'):
    print('Slack 환경변수 설정 완료!')
else:
    print('⚠️ Slack 토큰/채널이 비어 있습니다. slack_mcp_server.py는 dry-run 모드로 동작합니다.')


In [ ]:
import uuid
import httpx
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.checkpoint.memory import InMemorySaver
from stream_utils import stream_with_markdown


Tavily tool을 도구로 추가합니다.  
해당 서버를 터미널에서 실행해 주세요.

`web_search` 도구는 Tavily API의 주요 검색 옵션을 그대로 노출해, LLM이 질의 의도에 맞는 검색을 고를 수 있게 합니다.

- `topic`: `general` / `news` / `finance` 중 하나. 일반 웹 / 뉴스 / 금융 도메인으로 결과 풀을 좁힙니다.
- `time_range`: `day` / `week` / `month` / `year`. "오늘", "이번 주" 같은 최신성 요구가 있는 질의에 기간 필터를 겁니다 (생략 시 제한 없음).
- `include_raw_content`: `True`면 페이지 본문 전체를 결과에 포함합니다. 토큰 비용이 크므로 인용이나 정밀 분석이 꼭 필요할 때만 켭니다.


In [ ]:
%%writefile tavily_server.py
from mcp.server.fastmcp import FastMCP
from datetime import datetime
from typing import Literal
from dotenv import load_dotenv
from langchain_community.document_loaders import WebBaseLoader

mcp = FastMCP(
    name="tavily_search",
    instructions="Tavily 웹 검색 MCP 서버",
    port=8082
)

@mcp.resource("search://current_date")
def current_date() -> str:
    """현재 날짜를 반환합니다."""
    return f"오늘 날짜는 {datetime.now().isoformat()} 입니다."

@mcp.tool()
def fetch_url(url: str) -> str:
    """URL의 웹페이지 상세 내용을 가져옵니다. url: http(s):// 웹페이지 URL"""
    if not url.startswith(('http://', 'https://')):
        return f'오류: http(s):// URL만 지원합니다. 받은 값: {url}'
    try:
        docs = WebBaseLoader(url).load()
        content = docs[0].page_content if docs else '내용을 가져올 수 없습니다.'
        return content[:5000]
    except Exception as e:
        return f'fetch_url 오류: {type(e).__name__}: {e}'


@mcp.tool()
def web_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    time_range: Literal["day", "week", "month", "year"] | None = None,
    include_raw_content: bool = False,
) -> str:
    """
    query를 검색어로 하는 Tavily 웹 검색을 수행합니다.

    Args:
        query: 검색어
        max_results: 검색 결과 개수 (기본값: 5)
        topic: 검색 토픽. 'general'은 일반 웹, 'news'는 최근 뉴스 기사 중심,
            'finance'는 주식/시장/기업 재무 정보 중심으로 결과를 가져옵니다.
            (기본값: 'general')
        time_range: 결과를 최근 기간으로 제한합니다. 'day'(24시간), 'week'(7일),
            'month'(30일), 'year'(365일) 중 하나. None이면 기간 제한 없음.
            최신 정보가 필요한 질의(예: '오늘', '이번 주')에 사용하세요.
        include_raw_content: True면 각 결과에 페이지 본문 전체(raw_content)를
            포함합니다. 길어서 컨텍스트를 많이 차지하므로, 요약된 content로
            부족하고 본문 그대로의 인용이나 정밀 분석이 필요할 때만 True로 켜세요.
            (기본값: False)
    """
    from langchain_tavily import TavilySearch

    load_dotenv('.env', override=True)

    tavily_search = TavilySearch(
        max_results=max_results,
        topic=topic,
        time_range=time_range,
        include_raw_content=include_raw_content,
    )
    tavily_result = tavily_search.invoke(query)

    result = ""
    for item in tavily_result.get('results', []):
        result += f"Title: {item.get('title', 'N/A')}\n"
        result += f"URL: {item.get('url', 'N/A')}\n"
        if item.get('published_date'):
            result += f"Published: {item.get('published_date')}\n"
        result += f"Content: {item.get('content', 'N/A')}\n"
        if include_raw_content and item.get('raw_content'):
            result += f"Raw: {item.get('raw_content')}\n"
        result += "---\n"

    if not result:
        result = "검색 결과가 없습니다."

    return result

if __name__ == "__main__":
    print("🔍 Tavily 검색 MCP 서버 시작 (포트: 8082)")
    mcp.run(transport="streamable-http")

## 3. Slack MCP와 Elicitation

이번 실습에서는 Slack API를 MCP로 연결해 보겠습니다.

Slack MCP 서버 코드를 저장하고 실행합니다.

In [ ]:
%%writefile slack_mcp_server.py
"""
Slack MCP Server (FastMCP 기반)

기능:
  - get_channel_info       (Resource) 채널 정보 조회
  - get_channel_messages   (Tool)     최근 N시간 메시지 조회
  - post_message           (Tool)     채널에 메시지 전송 (채널 미지정 시 사용자에게 되묻기)
  - add_reaction           (Tool)     메시지에 이모지 리액션 추가
  - reply_to_thread        (Tool)     스레드에 답글 전송

특징:
  - 토큰이 없으면 자동 dry-run 모드 (더미 응답)
  - post_message는 채널 미지정 시 Elicitation으로 사용자에게 채널을 묻습니다.

실행: python slack_mcp_server.py
포트: 8084 (streamable-http)
"""

import os
from dataclasses import dataclass
from datetime import datetime, timezone, timedelta
from pathlib import Path

from dotenv import load_dotenv
from fastmcp import Context, FastMCP

# ─── 환경변수 로드와 dry-run 판정 ──────────────────────────

load_dotenv(Path(".env"))

SLACK_BOT_TOKEN = os.getenv("SLACK_BOT_TOKEN")
SLACK_CHANNEL_ID = os.getenv("SLACK_CHANNEL_ID")
DRY_RUN = not (SLACK_BOT_TOKEN and SLACK_CHANNEL_ID)

if DRY_RUN:
    print("⚠️  Slack 토큰이 없어 dry-run 모드로 시작합니다 (더미 응답).")
    SLACK_CHANNEL_ID = SLACK_CHANNEL_ID or "C_DRYRUN"

# ─── Slack 클라이언트와 FastMCP 서버 ────────────────────────

slack = None
SlackApiError = Exception
if not DRY_RUN:
    from slack_sdk import WebClient
    from slack_sdk.errors import SlackApiError  # noqa: F811
    slack = WebClient(token=SLACK_BOT_TOKEN)

mcp = FastMCP(
    name="slack",
    instructions=(
        "Slack 채널의 대화를 조회하고, 메시지를 보내고, 리액션을 추가하고, "
        "스레드에 답글을 다는 MCP 서버입니다."
    ),
)

# ─── 헬퍼 ────────────────────────────────────────────────

_user_cache: dict[str, str] = {}


def _resolve_user(user_id: str) -> str:
    """Slack user ID를 표시 이름으로 변환 (캐시 사용)."""
    if DRY_RUN:
        return f"user_{user_id}"
    if user_id not in _user_cache:
        try:
            info = slack.users_info(user=user_id)
            profile = info["user"]["profile"]
            _user_cache[user_id] = (
                profile.get("display_name") or profile.get("real_name") or user_id
            )
        except SlackApiError:
            _user_cache[user_id] = user_id
    return _user_cache[user_id]


def _fetch_messages(hours: int = 24) -> list[dict]:
    """Slack API로 최근 N시간 메시지를 가져옵니다."""
    if DRY_RUN:
        now = datetime.now(timezone.utc)
        return [
            {"user": "U001", "ts": str(now.timestamp() - 600), "text": "[DRY-RUN] 안녕하세요!"},
            {"user": "U002", "ts": str(now.timestamp() - 300), "text": "[DRY-RUN] 회의 자료 공유합니다."},
        ]

    oldest_ts = str((datetime.now(timezone.utc) - timedelta(hours=hours)).timestamp())
    all_messages: list[dict] = []
    cursor = None
    while True:
        kwargs = {"channel": SLACK_CHANNEL_ID, "oldest": oldest_ts, "limit": 200, "inclusive": True}
        if cursor:
            kwargs["cursor"] = cursor
        resp = slack.conversations_history(**kwargs)
        all_messages.extend(resp.get("messages", []))
        cursor = resp.get("response_metadata", {}).get("next_cursor")
        if not cursor:
            break
    all_messages.sort(key=lambda m: float(m.get("ts", "0")))
    return all_messages


def _format_messages(messages: list[dict]) -> str:
    lines = []
    for msg in messages:
        if msg.get("subtype") in ("channel_join", "channel_leave", "bot_message"):
            continue
        user_name = _resolve_user(msg.get("user", "unknown"))
        ts = float(msg.get("ts", "0"))
        dt = datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%Y-%m-%d %H:%M")
        text = msg.get("text", "").strip()
        if text:
            lines.append(f"[{dt}] (ts:{msg.get('ts','')}) {user_name}: {text}")
    return "\n".join(lines) if lines else "(메시지 없음)"


# ─── Resource ───────────────────────────────────────────

@mcp.resource("info://channel")
def get_channel_info() -> str:
    """연결된 Slack 채널의 이름, ID, 멤버 수, 주제, 목적을 반환합니다."""
    if DRY_RUN:
        return (
            f"채널 정보 (DRY-RUN):\n"
            f"  이름: #dry-run-channel\n"
            f"  ID: {SLACK_CHANNEL_ID}\n"
            f"  멤버 수: 0\n"
            f"  주제: (dry-run)\n"
            f"  목적: (dry-run)"
        )
    try:
        resp = slack.conversations_info(channel=SLACK_CHANNEL_ID)
        ch = resp["channel"]
        return (
            f"채널 정보:\n"
            f"  이름: #{ch.get('name', 'unknown')}\n"
            f"  ID: {SLACK_CHANNEL_ID}\n"
            f"  멤버 수: {ch.get('num_members', '?')}\n"
            f"  주제: {ch.get('topic', {}).get('value') or '(없음)'}\n"
            f"  목적: {ch.get('purpose', {}).get('value') or '(없음)'}"
        )
    except SlackApiError as e:
        return f"채널 정보를 가져올 수 없습니다: {e.response['error']}"


# ─── Tool ───────────────────────────────────────────────

@mcp.tool()
def get_channel_messages(hours: int = 24) -> str:
    """Slack 채널에서 최근 N시간 동안의 메시지를 가져옵니다.

    Args:
        hours: 몇 시간 이내의 메시지를 가져올지 (기본값: 24시간)

    Returns:
        시간순으로 정렬된 대화 내용
    """
    try:
        messages = _fetch_messages(hours=hours)
        if not messages:
            return f"최근 {hours}시간 동안 메시지가 없습니다."
        formatted = _format_messages(messages)
        count = len([
            m for m in messages
            if m.get("subtype") not in ("channel_join", "channel_leave", "bot_message")
        ])
        prefix = "[DRY-RUN] " if DRY_RUN else ""
        return f"{prefix}최근 {hours}시간 메시지 ({count}개)\n{'━' * 40}\n{formatted}"
    except SlackApiError as e:
        return f"메시지를 가져오지 못했습니다: {e.response['error']}"


@dataclass
class TargetChannel:
    """Elicitation 응답 스키마: 사용자가 어느 채널에 보낼지 선택합니다."""
    channel_id: str


@mcp.tool()
async def post_message(message: str, ctx: Context, target_channel: str | None = None) -> str:
    """Slack 채널에 메시지를 전송합니다.

    target_channel을 지정하지 않으면, MCP Elicitation으로 사용자에게 어느 채널에
    보낼지 묻습니다.

    Args:
        message: 전송할 메시지 내용
        target_channel: 보낼 채널 ID. 비어있으면 사용자에게 묻습니다.

    Returns:
        전송 결과
    """
    chosen = target_channel
    if not chosen:
        result = await ctx.elicit(
            message=(
                f'어느 채널에 메시지를 보낼까요?\n\n'
                f'  메시지: "{message[:80]}..."\n\n'
                f'채널 ID를 입력하세요 (기본: {SLACK_CHANNEL_ID}).'
            ),
            response_type=TargetChannel,
        )
        if result.action == "accept":
            chosen = result.data.channel_id or SLACK_CHANNEL_ID
        elif result.action == "decline":
            return "사용자가 채널 선택을 거절하여 메시지를 보내지 않았습니다."
        else:
            return "사용자가 작업을 취소했습니다."

    if DRY_RUN:
        return f"[DRY-RUN] 메시지가 {chosen} 채널에 전송된 것으로 처리되었습니다. (ts: 0.0)"

    try:
        resp = slack.chat_postMessage(channel=chosen, text=message)
        return f"메시지가 {chosen}에 전송되었습니다. (ts: {resp.get('ts', '')})"
    except SlackApiError as e:
        return f"메시지 전송 실패: {e.response['error']}"


@mcp.tool()
def add_reaction(message_ts: str, emoji: str) -> str:
    """Slack 메시지에 이모지 리액션을 추가합니다.

    Args:
        message_ts: 리액션을 달 메시지의 타임스탬프
        emoji: 이모지 이름 (예: "thumbsup", "heart", "fire", "eyes")

    Returns:
        리액션 추가 결과
    """
    if DRY_RUN:
        return f"[DRY-RUN] :{emoji}: 리액션이 메시지(ts:{message_ts})에 추가된 것으로 처리되었습니다."
    try:
        slack.reactions_add(channel=SLACK_CHANNEL_ID, timestamp=message_ts, name=emoji)
        return f":{emoji}: 리액션이 메시지(ts:{message_ts})에 추가되었습니다."
    except SlackApiError as e:
        return f"리액션 추가 실패: {e.response['error']}"


@mcp.tool()
def reply_to_thread(thread_ts: str, message: str) -> str:
    """Slack 메시지의 스레드에 답글을 전송합니다.

    Args:
        thread_ts: 답글을 달 원본 메시지의 타임스탬프
        message: 답글 내용

    Returns:
        답글 전송 결과
    """
    if DRY_RUN:
        return f"[DRY-RUN] 스레드 답글이 ts:{thread_ts}에 전송된 것으로 처리되었습니다."
    try:
        resp = slack.chat_postMessage(
            channel=SLACK_CHANNEL_ID, text=message, thread_ts=thread_ts
        )
        return f"스레드 답글이 전송되었습니다. (ts: {resp.get('ts', '')})"
    except SlackApiError as e:
        return f"스레드 답글 전송 실패: {e.response['error']}"


# ─── 서버 실행 ──────────────────────────────────────────

if __name__ == "__main__":
    import sys, io
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8", errors="replace")
    sys.stderr = io.TextIOWrapper(sys.stderr.buffer, encoding="utf-8", errors="replace")

    dryrun_label = "  *DRY-RUN*" if DRY_RUN else ""
    print(f"Slack MCP 서버 시작 (포트: 8084{dryrun_label})")
    print(f"  채널 ID: {SLACK_CHANNEL_ID}")
    print(f"  URL: http://localhost:8084/mcp")
    mcp.run(transport="streamable-http", port=8084)


### Slack MCP 서버 실행하기

Slack MCP 서버도 별도 터미널에서 실행합니다.

> 터미널 열기: VS Code에서 `Ctrl+J` 또는 별도 cmd/터미널 창
>
> ```bash
> python slack_mcp_server.py
> ```
>
> 서버가 시작되면 `http://localhost:8084/mcp`에서 접속 가능합니다.
> 토큰이 없으면 시작 로그에 `*DRY-RUN*` 표시가 붙습니다.

서버가 실행 중인 상태에서 아래 셀을 실행하세요.


In [ ]:
# Slack MCP 서버 헬스체크
import httpx

try:
    r = httpx.post(
        'http://localhost:8084/mcp',
        headers={'Content-Type': 'application/json',
                 'Accept': 'application/json, text/event-stream'},
        json={'jsonrpc': '2.0', 'id': 0, 'method': 'initialize',
              'params': {'protocolVersion': '2025-06-18', 'capabilities': {},
                         'clientInfo': {'name': 'healthcheck', 'version': '0'}}},
        timeout=3.0,
    )
    print('Slack MCP 서버 응답 OK:', r.status_code)
except httpx.ConnectError:
    print('서버에 연결할 수 없습니다. `python slack_mcp_server.py`를 다른 터미널에서 먼저 실행하세요.')

### Elicitation

Elicitation은 서버가 작업 도중에 사용자에게 추가 정보를 되묻는 기능입니다.

예를 들어 사용자가 "메시지를 보내줘"라고만 하고 채널을 지정하지 않으면,  
서버가 "어느 채널에 보낼까요?"라고 다시 물어볼 수 있습니다.  


### Slack Agent 연결 + Elicitation 콜백 등록

서버가 정보를 되물을 때 클라이언트가 어떻게 답할지를 on_elicitation 콜백으로 정합니다.  
여기서는 데모를 위해 환경변수의 채널로 자동 응답하고, 뒤에서 사용자가 직접 입력하는 버전으로 바꿉니다.


In [ ]:
# 서버가 사용자에게 정보를 되물을 때(elicitation), 클라이언트가 어떻게 답할지 정해줍니다.
from langchain_mcp_adapters.callbacks import Callbacks
from mcp.types import ElicitResult

# 데모용으로 자동 응답합니다.
# 실제 IDE 클라이언트(Cursor, Claude Desktop 등)에서는 사용자에게 입력 폼을 표시합니다.
async def on_elicitation(mcp_context, params, context):
    print(f'  ⚡ 서버가 elicit 요청: {params.message}...')
    print(f'     요청 스키마: {params.requestedSchema.get("required", [])}')

    auto_channel = os.environ.get('SLACK_CHANNEL_ID', 'C_DRYRUN')
    print(f'     자동 응답: channel_id={auto_channel}')
    return ElicitResult(action='accept', content={'channel_id': auto_channel})


slack_client = MultiServerMCPClient(
    {'slack': {'transport': 'http', 'url': 'http://localhost:8084/mcp'}},
    callbacks=Callbacks(on_elicitation=on_elicitation),
)

slack_tools = await slack_client.get_tools()
print(f'\nSlack 도구 수: {len(slack_tools)}')
for t in slack_tools:
    print(f'  - {t.name}: {t.description}')

아래에서 에이전트를 만들고 일반 요청을 한 번 보냅니다.  
빠진 정보가 없는 요청이라 서버가 되묻지 않고 콜백도 호출되지 않습니다.  
elicitation은 다음 데모에서 일부러 채널을 비워 발생시킵니다.


In [ ]:
slack_agent = create_agent(
    'gpt-5.2',
    tools=slack_tools,
    system_prompt='''당신은 Slack 채널 관리 AI 어시스턴트입니다.
채널의 대화를 조회하고, 메시지를 보내고, 리액션을 추가할 수 있습니다.
항상 한국어로 응답하세요.'''
)

await stream_with_markdown(slack_agent, '최근 1시간 동안의 메시지를 가져와서 요약해줘.', max_tool_result=1000)

### 컨텍스트가 얼마나 쌓이는가

도구 정의는 매 요청에 고정 오버헤드로 들어가고, 도구 호출이 이어질수록 입력 컨텍스트가 호출마다 커집니다.  
stream_with_markdown은 문자열만 반환하므로, 토큰을 보려면 invoke로 messages를 받아 usage_metadata를 확인합니다.  
통합 봇은 서버가 셋이라 도구 정의가 더 불어납니다.


In [ ]:
from langchain.messages import AIMessage, ToolMessage
import json

# 도구 정의 자체가 매 요청의 고정 오버헤드로 들어간다.
def schema_chars(t):
    sch = getattr(t, 'args_schema', None)
    js = sch.model_json_schema() if hasattr(sch, 'model_json_schema') else {}
    return len(t.name) + len(t.description or '') + len(json.dumps(js, ensure_ascii=False))

overhead = sum(schema_chars(t) for t in slack_tools)
print(f'도구 {len(slack_tools)}개의 정의 분량: 약 {overhead:,}자')

# 도구 호출이 이어질수록 입력 컨텍스트가 커진다.
result = slack_agent.invoke(
    {'messages': [{'role': 'user', 'content': '최근 1시간 메시지를 가져와서 요약해줘.'}]}
)
for i, m in enumerate([m for m in result['messages'] if isinstance(m, AIMessage) and m.usage_metadata], 1):
    u = m.usage_metadata
    print(f'LLM 호출 {i}: 입력 {u["input_tokens"]:>6} 토큰, 출력 {u["output_tokens"]:>5} 토큰')

tool_msgs = [m for m in result['messages'] if isinstance(m, ToolMessage)]
print(f'도구 결과 {len(tool_msgs)}개, 누적 {sum(len(str(m.content)) for m in tool_msgs):,}자')

### Elicitation 데모: 채널을 지정하지 않고 메시지 보내기

`post_message` 도구는 채널이 비어 있으면 서버가 사용자에게 어느 채널에 보낼지 되묻습니다.  
위에서 등록한 `on_elicitation` 콜백이 자동으로 답하기 때문에, 콘솔에 `⚡ 서버가 elicit 요청...` 로그가 출력되고 흐름이 이어집니다.

> 실제 IDE 클라이언트라면 이 시점에 사용자에게 입력 폼이 나타나 직접 채널을 선택하게 됩니다.


In [ ]:
# 메시지 전송: 채널 명시
await stream_with_markdown(slack_agent, '"elicitation 데모입니다"라고 보내줘.', max_tool_result=1000)

### 사용자가 직접 입력하게 만들기 (`input()`)

위에서는 콜백이 자동으로 채널 ID를 채웠습니다.  
실제 IDE 클라이언트(Cursor, Claude Desktop 등)에서는 사용자에게 입력 폼이 나타나 직접 채널을 고릅니다.

Jupyter 노트북에서도 비슷하게 동작하게 만들 수 있습니다.  
콜백 안에서 `input()`을 호출하면 셀 출력 영역 아래에 텍스트 입력 상자가 나타나고, 사용자가 입력 후 Enter를 누를 때까지 대기합니다.

비동기 함수에서는 `asyncio.to_thread`로 감싸서 이벤트 루프를 막지 않게 합니다.


In [ ]:
# input() 기반 콜백: 사용자가 직접 채널 ID를 타이핑하게 합니다.
import asyncio

async def on_elicitation_interactive(mcp_context, params, context):
    print(f'⚡ 서버가 요청: {params.message}')
    required = params.requestedSchema.get('required', [])

    answers = {}
    for field in required:
        # asyncio.to_thread로 input()을 별도 스레드에서 실행 (이벤트 루프 보호)
        value = await asyncio.to_thread(input, f'  → {field} 입력 (cancel 입력 시 취소): ')
        if value.strip().lower() == 'cancel':
            return ElicitResult(action='cancel')
        answers[field] = value.strip()

    return ElicitResult(action='accept', content=answers)


# 새 콜백을 적용한 클라이언트와 에이전트를 만듭니다.
slack_client_interactive = MultiServerMCPClient(
    {'slack': {'transport': 'http', 'url': 'http://localhost:8084/mcp'}},
    callbacks=Callbacks(on_elicitation=on_elicitation_interactive),
)
slack_tools_interactive = await slack_client_interactive.get_tools()

slack_agent_interactive = create_agent(
    'gpt-5.2',
    tools=slack_tools_interactive,
    system_prompt='매 도구 사용마다 중간 과정을 설명하세요.'
)

# # 같은 요청을 다시 보냅니다. 이번엔 셀 아래에 진짜 입력 상자가 뜹니다.
# result = await slack_agent_interactive.ainvoke({
#     'messages': [{'role': 'user', 'content': '아무튼 어디든 좋으니 "직접 입력 elicitation 데모"라고 보내줘.'}]
# })
# print('\n--- 최종 응답 ---')
# print(result['messages'][-1].text)

In [ ]:
# Elicitation 작동 뒤 재개
await stream_with_markdown(slack_agent_interactive, '아무튼 어디든 좋으니 "직접 입력 elicitation 데모"라고 보내줘.')

## 4. Stateless vs Stateful MCP

MCP는 클라이언트가 서버에 한 번 `initialize`로 세션을 연 뒤, 그 안에서 도구 호출이 이어지는 프로토콜입니다.  
Streamable HTTP 트랜스포트는 초기화 응답의 `Mcp-Session-Id` 헤더로 후속 요청을 같은 세션에 묶고,  
stdio 트랜스포트에서는 자식 프로세스의 표준 입출력 파이프 자체가 하나의 세션을 형성합니다.

- Stateless MCP: 도구 호출 사이에 서버가 보존하는 상태가 없는 서버입니다. 산술 계산, 단발성 검색, 파일 단건 읽기처럼  
 매 호출이 독립적이라, 세션이 새로 열려도 결과가 같습니다.
- Stateful MCP: 브라우저, DB 커서, 인증된 API 핸들처럼 호출 사이에 서버 상태가 누적되는 서버입니다.  
  두 호출이 같은 세션으로 라우팅되어야 합니다.


이에 따라, `langchain-mcp-adapters` 두 가지 방식으로 구현합니다.
- Stateless는 `client.get_tools()`만으로 충분합니다. 이 헬퍼는 도구 호출마다 임시 세션을 열고 닫아 정리합니다.
- Stateful은 `async with client.session(...)`로 세션을 먼저 연 뒤,  
 `load_mcp_tools(session)`으로 같은 세션에 묶인 도구를 로드해야 누적 상태가 호출 사이에서 유지됩니다.


### Playwright MCP: 세션 단위 상태가 필요한 대표 사례

Playwright는 웹 자동화를 위해, 브라우저를 조작하는 MCP입니다.   
해당 MCP는 대표적인 stateful 서버입니다.    

`browser_navigate`로 페이지를 이동한 뒤 `browser_snapshot`을 호출하려면,    
두 호출이 동일한 브라우저 인스턴스를 가리켜야 합니다.    
즉 두 호출이 동일한 MCP 클라이언트 세션 안에서, 동일한 서버 프로세스에 도달해야 합니다.


In [1]:
%%writefile advanced_mcp_example.py
"""
Playwright MCP를 stateful 세션으로 사용하는 예제.
- client.session() 컨텍스트로 모든 도구가 동일한 ClientSession을 공유
- 이렇게 하면 navigate -> snapshot 사이에 브라우저 상태가 유지됨
"""

import asyncio
import sys
import os

from dotenv import load_dotenv
from stream_utils import stream_print

load_dotenv(".env", override=True)

from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools   # ← 추가
from langchain.agents import create_agent

# .venv의 python 경로 (Windows에서는 Scripts\\python.exe, macOS/Linux에서는 bin/python)
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
if sys.platform == "win32":
    PYTHON = os.path.join(BASE_DIR, ".venv", "Scripts", "python.exe")
else:
    PYTHON = os.path.join(BASE_DIR, ".venv", "bin", "python")


async def main():
    # STDIO 방식으로 MCP 서버 연결
    client = MultiServerMCPClient({
        "playwright": {
            "transport": "stdio",
            "command": "npx",
            "args": ["@playwright/mcp@latest", "--headless"],
            # headless : 실제 브라우저 창을 열지 않음
        }
    })

    # 명시적 세션 컨텍스트.
    # 기본 client.get_tools()는 호출마다 새 세션을 만들어 stateful 서버에서 상태가 유실
    # 아래 패턴은 한 세션을 열고 그 안에서 도구를 로드 -> 모든 도구 호출이 같은 세션을 공유
    async with client.session("playwright") as session:
        # 같은 세션에 바인딩된 도구 로드
        tools = await load_mcp_tools(session)

        print(f"로드된 도구 수: {len(tools)}")
        for t in tools:
            print(f"  - {t.name}: {t.description}")

        agent = create_agent(
            "gpt-5.2",
            tools=tools,
            system_prompt="도구를 사용할 때마다, 중간 과정을 매번 간략히 설명하세요.",
        )

        question = "https://sudoremove.com/news/ 에 접속해서, 가장 최신 날짜 뉴스 내용 설명해줘"


        result = await stream_print(agent, question)


if __name__ == "__main__":
    asyncio.run(main())

Writing advanced_mcp_example.py


터미널에서 해당 스크립트를 실행하고 결과를 확인하세요.

---
## 이미지 생성 MCP 서버

텍스트로 그림을 만드는 MCP 서버를 추가합니다.  
`generate_image` 도구가 gpt-image-2로 이미지를 생성해 `outputs/generated_images/`에 저장하고 저장 경로를 돌려줍니다.  
stdio 트랜스포트라 클라이언트가 필요할 때 이 스크립트를 직접 spawn하므로, 별도 터미널로 실행할 필요가 없습니다.


In [ ]:
%%writefile image_server.py
"""image_server.py

GPT-Image-2 이미지 생성 MCP 서버입니다.
generate_image(prompt) 도구가 이미지를 생성해 outputs/generated_images/에 저장하고,
저장 경로를 돌려줍니다. 슬랙 업로드는 봇(slack_app)이 맡습니다.

STDIO 트랜스포트라, 클라이언트(build_agent)가 필요할 때 이 스크립트를 직접 spawn합니다.
streamable-http 서버들과 달리 별도 터미널로 실행해둘 필요가 없습니다.
"""

import base64
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

from dotenv import load_dotenv
from fastmcp import FastMCP
from langchain_openai import ChatOpenAI

load_dotenv(".env", override=True)

mcp = FastMCP(
    name="image",
    instructions="텍스트 프롬프트로 이미지를 생성하는 MCP 서버입니다.",
)


@mcp.tool()
def generate_image(prompt: str) -> str:
    """텍스트 프롬프트를 기반으로 AI 이미지를 생성하고 로컬에 저장합니다.

    이미지 생성, 그림 그리기, 시각적 콘텐츠 제작 요청에 사용합니다.
    내부적으로 GPT 이미지 생성 모델(gpt-image-2)을 호출합니다.

    Args:
        prompt: 생성할 이미지를 설명하는 텍스트. 구체적이고 상세할수록 품질이 좋아집니다.

    Returns:
        저장 경로를 담은 메시지. 예: "✅ Image saved: outputs/generated_images/image_....png"
    """
    llm = ChatOpenAI(model="gpt-5.2")
    image_tool = {"type": "image_generation", "model": "gpt-image-2"}
    ai_message = llm.bind_tools([image_tool]).invoke(prompt)

    image = next(item for item in ai_message.content_blocks if item["type"] == "image")

    ts = datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y%m%d_%H%M%S_%f")[:-3]
    out_dir = Path("outputs") / "generated_images"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"image_{ts}.png"
    out_path.write_bytes(base64.b64decode(image["base64"]))

    return f"✅ Image saved: {out_path.as_posix()}"


if __name__ == "__main__":
    # STDIO 트랜스포트: stdout이 MCP 프로토콜 채널이므로 로그는 stderr로 보낸다.
    import sys

    print("🎨 이미지 생성 MCP 서버 시작 (stdio)", file=sys.stderr)
    mcp.run(transport="stdio")


---
## MCP 설정을 JSON으로 분리하기

`MultiServerMCPClient`에 넣던 서버 설정을 `mcp_servers.json`으로 분리합니다. Claude Desktop, Cursor 같은 클라이언트가 쓰는 mcpServers 형식과 같습니다.

서버마다 stateful 여부를 표시합니다. tavily, slack, image는 호출마다 임시 세션으로 도구를 받는 stateless 서버이고, Playwright는 브라우저 세션을 유지해야 하는 stateful 서버입니다.

In [ ]:
%%writefile mcp_servers.json
{
  "mcpServers": {
    "tavily": {
      "transport": "streamable_http",
      "url": "http://localhost:8082/mcp"
    },
    "slack": {
      "transport": "streamable_http",
      "url": "http://localhost:8084/mcp"
    },
    "image": {
      "transport": "stdio",
      "command": "python3",
      "args": ["image_server.py"]
    },
    "playwright": {
      "transport": "stdio",
      "command": "npx",
      "args": ["@playwright/mcp@latest", "--headless"],
      "stateful": true
    }
  }
}


`mcp_config`가 JSON을 읽어 stateful 여부로 갈라 줍니다. stdio 서버의 python 커맨드는 현재 인터프리터로 치환해, 가상환경 활성화 여부와 무관하게 같은 파이썬으로 spawn되게 합니다.

In [ ]:
%%writefile mcp_config.py
"""mcp_config.py

MCP 서버 설정을 mcp_servers.json에서 읽어 옵니다. JSON은 MCP 생태계 표준과 같은
mcpServers 형식이며, 서버별 "stateful": true 플래그로 지속 세션이 필요한 서버를
표시합니다.

Claude Code 같은 클라이언트는 모든 서버에 시작 시 세션을 한 번 열어 유지하므로 이
구분이 없습니다. 우리 build_agent는 stateless 서버를 get_tools로 가볍게(호출마다
임시 세션) 로드하고, stateful 서버(예: Playwright)는 slack_app이 세션으로 유지하므로
둘을 구분합니다.

사용 예시
    from mcp_config import load_servers

    stateless = load_servers(stateful=False)  # build_agent가 get_tools로 로드
    stateful = load_servers(stateful=True)    # slack_app이 세션으로 유지
"""

from __future__ import annotations

import json
import sys
from pathlib import Path

CONFIG_PATH = Path(__file__).parent / "mcp_servers.json"

# stdio 서버의 python 커맨드를 현재 인터프리터로 치환할 때 인식하는 이름.
_PYTHON_ALIASES = {"python", "python3"}


def load_servers(stateful: bool = False, path=None) -> dict:
    """mcp_servers.json에서 서버 설정을 읽어 stateful 여부로 거른 dict를 돌려줍니다.

    반환되는 connection dict에서는 확장 키 'stateful'을 제거해, MultiServerMCPClient에
    그대로 넣을 수 있는 표준 형태로 만듭니다. 파일이 없으면 빈 dict를 돌려줍니다.

    Args:
        stateful: True면 지속 세션이 필요한 서버만, False면 stateless 서버만 반환.
        path: JSON 경로 override (기본 mcp_servers.json).

    Returns:
        {서버이름: connection dict} 형태의 dict.
    """
    cfg_path = Path(path) if path else CONFIG_PATH
    if not cfg_path.exists():
        return {}
    data = json.loads(cfg_path.read_text(encoding="utf-8"))
    servers = data.get("mcpServers", {})
    out = {}
    for name, conn in servers.items():
        if bool(conn.get("stateful", False)) != stateful:
            continue
        clean = {k: v for k, v in conn.items() if k != "stateful"}
        # stdio 서버의 python 커맨드는 부모와 같은 인터프리터(sys.executable)로 치환한다.
        # JSON엔 "python3"라는 포터블한 문자열을 두되, 실제 spawn은 deps가 보장된 현재
        # 인터프리터로 가게 해 PATH(venv 활성화 여부) 의존을 없앤다. npx 등은 그대로 둔다.
        if clean.get("command") in _PYTHON_ALIASES:
            clean["command"] = sys.executable
        out[name] = clean
    return out


---
## build_agent로 에이전트 조립하기

모델, 표준 도구, MCP 서버를 `build_agent.py` 한 곳에서 조립합니다.  
MCP 서버는 `mcp_config`로 `mcp_servers.json`에서 읽어, stateless 서버 도구를 표준 도구에 더합니다.  
stateful 서버(Playwright)는 호출자가 `extra_tools`로 넘기고, `post_message`가 채널을 되물을 때를 대비해 elicitation 콜백을 등록합니다.  
MCP 도구를 await로 받으므로 async 함수입니다.


In [ ]:
%%writefile build_agent.py
"""build_agent.py

에이전트를 한 곳에서 조립하는 팩토리입니다. 모델은 select_model로 고르고,
표준 도구(tools.py)에 MCP 서버 도구를 더해 에이전트를 만듭니다. 서비스 코드(예: Slack 봇)는
build_agent()만 호출해 같은 에이전트를 받습니다.

MCP 도구를 await로 받아오므로 build_agent는 async 함수입니다.
"""

from __future__ import annotations

import os

from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

from select_model import load_model
from tools import DEFAULT_TOOLS
from mcp_config import load_servers

DEFAULT_SYSTEM_PROMPT = (
    '당신은 유용한 AI 어시스턴트입니다. '
    '필요하면 도구를 사용하고, 도구를 실행하기 전 중간 과정을 간략히 설명하세요.'
)

# build_agent가 직접 연결하는 MCP 서버. mcp_servers.json의 stateless 서버를 읽는다.
# stateful 세션(Playwright 등)은 봇이 세션으로 열어 extra_tools로 넘긴다.
MCP_SERVERS = load_servers(stateful=False)


async def _auto_elicitation(mcp_context, params, context):
    """Slack MCP의 post_message가 채널을 되물을 때 기본 채널로 자동 응답합니다.

    봇이나 노트북에는 사용자에게 입력 폼을 표시할 방법이 없으므로, SLACK_CHANNEL_ID
    환경변수의 채널로 답합니다.
    """
    from mcp.types import ElicitResult

    channel = os.environ.get('SLACK_CHANNEL_ID', 'C_DRYRUN')
    return ElicitResult(action='accept', content={'channel_id': channel})


async def load_mcp_tools(servers=None):
    """MCP 서버에서 도구를 받아옵니다. 연결에 실패하면 빈 목록을 돌려줍니다."""
    servers = MCP_SERVERS if servers is None else servers
    if not servers:
        return []
    try:
        from langchain_mcp_adapters.callbacks import Callbacks

        client = MultiServerMCPClient(
            servers, callbacks=Callbacks(on_elicitation=_auto_elicitation)
        )
        return await client.get_tools()
    except Exception as e:
        print(f'[build_agent] MCP 도구 로드 실패, 표준 도구만 사용합니다: {e}')
        return []


async def build_agent(
    model_name=None,
    platform='openai',
    model=None,
    tools=None,
    extra_tools=None,
    middleware=None,
    system_prompt=None,
    checkpointer=None,
    mcp_servers=None,
    **model_kwargs,
):
    """현재까지 구성된 에이전트를 만들어 반환합니다.

    Args:
        model_name: 모델 이름. select_model.load_model로 전달됩니다.
        platform: 'openai' | 'vllm' | 'ollama'. load_model로 전달됩니다.
        model: 이미 만든 chat model. 주면 model_name/platform은 무시됩니다.
        tools: 표준 도구 대신 쓸 도구 목록. 생략 시 tools.py의 DEFAULT_TOOLS.
        extra_tools: 호출자가 직접 만든 도구를 추가로 붙입니다. 봇이 stateful
            세션(예: Playwright)에서 받은 도구를 넘길 때 씁니다.
        middleware: create_agent에 넘길 미들웨어 목록. 노트북에서 만든
            ContextInjectionMiddleware / SkillMiddleware /
            EpisodicMemoryMiddleware / HumanInTheLoopMiddleware를
            여기에 넣으면 그 능력이 그대로 봇에 얹힙니다.
        system_prompt: 시스템 프롬프트. 생략 시 기본값을 씁니다.
        checkpointer: 멀티턴 대화를 위한 체크포인터.
        mcp_servers: 연결할 MCP 서버 설정. 생략 시 MCP_SERVERS, {}면 MCP를 끕니다.
        **model_kwargs: load_model로 전달되는 추가 인자 (reasoning_effort 등).
    """
    if model is None:
        model = load_model(model_name, platform=platform, **model_kwargs)
    base_tools = list(DEFAULT_TOOLS) if tools is None else list(tools)
    mcp_tools = await load_mcp_tools(mcp_servers)
    extra = list(extra_tools) if extra_tools else []
    if system_prompt is None:
        system_prompt = DEFAULT_SYSTEM_PROMPT
    return create_agent(
        model,
        tools=base_tools + mcp_tools + extra,
        middleware=list(middleware) if middleware else [],
        system_prompt=system_prompt,
        checkpointer=checkpointer,
    )

tavily, slack 서버가 실행 중인 상태에서 불러와 실행합니다. image는 stdio라 호출 시 자동 spawn됩니다.


In [ ]:
from build_agent import build_agent

agent = await build_agent()

result = await agent.ainvoke({
    'messages': [{'role': 'user', 'content': '최근 1시간 슬랙 메시지를 가져와서 한 줄로 요약해줘.'}]
})
print(result['messages'][-1].text)

---
## 통합 에이전트를 Slack 봇으로 올리기

build_agent로 조립한 에이전트를 `slack_app.py`가 Slack에 올립니다. slack_app은 `build_agent()`로 에이전트를 받아 멘션에 응답합니다.

동작:
- 채널에서 봇을 멘션하면 스레드를 열어 응답하고, 그 스레드 안에서는 멘션 없이 이어집니다. 멘션 없는 일반 채널 대화는 무시합니다.
- 응답은 `astream_events`로 토큰과 도구 호출을 스레드에 점진 게시합니다.
- `generate_image`로 만든 그림은 스레드에 파일로 첨부합니다.
- 스레드 대화는 SQLite 체크포인터에 저장되어 봇을 껐다 켜도 이어집니다.
- Playwright는 봇이 세션을 열어 유지하고 그 도구를 build_agent에 넘깁니다.

Slack 토큰과 스코프, 이벤트 구독 설정은 `slack_setup.md`를 참고하세요.

In [ ]:
%%writefile slack_app.py
"""slack_app.py

build_agent로 만든 에이전트를 Slack에 올리는 배포 하네스입니다.
채널에서 봇을 @멘션하면 스레드를 열어 응답하고, 그 스레드 안에서는 멘션 없이도
대화가 이어집니다(멘션 없는 일반 채널 대화는 무시). 응답은 astream_events로 토큰과
도구 호출을 스레드에 점진 게시하고, generate_image로 만든 그림은 파일로 첨부합니다.
스레드 대화는 SQLite 체크포인터에 저장되어 봇을 껐다 켜도 이어집니다.

모델, 표준 도구, MCP 도구는 build_agent가 조립합니다. 이 파일은 그것들을 직접
알지 않으므로, build_agent.py가 갱신되면 이 봇도 같은 에이전트를 그대로 씁니다.

전제 조건:
  MCP 서버 설정은 mcp_servers.json에서 읽습니다. stateless 서버(tavily, slack,
  image)는 build_agent가 로드하고, stateful 서버(Playwright)는 이 봇이 세션으로
  유지합니다. 서버가 실행 중이지 않으면 그 도구만 제외되고 나머지로 동작합니다.
  Playwright는 stdio로 npx가 자체 실행하며, 브라우저(Chrome)가 있어야 합니다.

Slack 설정 (slack_setup.md 참고):
  Event Subscriptions에 message.channels(스레드 메시지 수신)와 app_mention을 구독하고,
  봇 스코프에 files:write(이미지 첨부)를 추가하세요.

필요한 환경변수 (.env):
  OPENAI_API_KEY
  SLACK_BOT_TOKEN   (xoxb-...)  Bot User OAuth Token
  SLACK_APP_TOKEN   (xapp-...)  App-Level Token (Socket Mode 활성화 필요)

추가 설치:
  pip install slack_bolt

실행:
  python slack_app.py             # = --mode compact (기본)
  python slack_app.py --mode 1    # AI 텍스트만
  python slack_app.py --mode 3    # 전체 풀 출력
"""

import argparse
import asyncio
import contextlib
import json
import os
import re
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(".env", override=True)

from slack_bolt.async_app import AsyncApp
from slack_bolt.adapter.socket_mode.async_handler import AsyncSocketModeHandler

from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools

from build_agent import build_agent
from mcp_config import load_servers

# 스레드 대화를 파일에 저장해 봇을 껐다 켜도 이어집니다. thread_id = slack_{thread_ts}.
CHECKPOINT_DB = "slack_threads.db"


SLACK_SYSTEM_PROMPT = """당신은 Slack에서 동작하는 AI 어시스턴트입니다.

규칙:
  1) 항상 한국어로 응답하며, 도구를 사용할 때는 사용하기 전 중간 과정을 간략히 설명합니다.
  2) Slack 관련 도구를 호출할 때는 사용자 입력 앞에 주어진 [현재 Slack 컨텍스트]의
     채널 ID와 스레드 ts를 그대로 사용하세요. 채널을 추측해서 보내지 마세요.
  3) 도구를 여러 개 조합해도 됩니다. 검색 결과만으로 부족하면 브라우저 도구로 페이지를 열어 확인하세요.
  4) generate_image로 만든 그림은 시스템이 자동으로 이 스레드에 파일로 첨부합니다.
     사용자에게 직접 올리라고 안내하지 말고, 그림을 첨부했다고 자연스럽게 답하세요.
"""

# ─── 출력 모드 파싱 ─────────────────────────────────────────

MODE_ALIASES = {"1": "final", "2": "compact", "3": "verbose"}
VALID_MODES = ("final", "compact", "verbose")


def parse_args() -> str:
    p = argparse.ArgumentParser(
        prog="slack_app",
        description="build_agent 에이전트를 Slack에 올리는 봇 (스트리밍 응답)",
    )
    p.add_argument(
        "--mode", "-m",
        default="compact",
        choices=list(MODE_ALIASES.keys()) + list(VALID_MODES),
        help="출력 모드. 1/final = AI 텍스트만, 2/compact = AI + 도구 한 줄 요약 (기본), "
             "3/verbose = 전체 풀 출력",
    )
    args = p.parse_args()
    return MODE_ALIASES.get(args.mode, args.mode)


# ─── Slack 스트리밍 헬퍼 ─────────────────────────────────────

def _format_args_full(data, limit: int = 400) -> str:
    """on_tool_start의 event['data']를 코드블록 친화적으로 직렬화 (verbose)."""
    try:
        s = json.dumps(data, ensure_ascii=False, indent=2, default=str)
    except (TypeError, ValueError):
        s = str(data)
    return s if len(s) <= limit else s[:limit] + "  …(생략)"


def _compact_args(data, limit: int = 120) -> str:
    """on_tool_start의 event['data']에서 args를 한 줄 key=value 요약 (compact)."""
    inp = data.get("input", data) if isinstance(data, dict) else data
    if isinstance(inp, dict):
        parts = []
        for k, v in inp.items():
            sv = repr(v) if isinstance(v, str) else str(v)
            sv = sv.replace("\n", " ")
            if len(sv) > 40:
                sv = sv[:40] + "…"
            parts.append(f"{k}={sv}")
        s = ", ".join(parts)
    else:
        s = str(inp).replace("\n", " ")
    return s if len(s) <= limit else s[:limit] + "…"


def _stringify_content(content) -> str:
    """ToolMessage.content가 str / content-block list / 그 외 형태로 와도 str로 정규화한다."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for blk in content:
            if isinstance(blk, dict):
                parts.append(blk.get("text") or blk.get("content") or json.dumps(blk, ensure_ascii=False, default=str))
            else:
                parts.append(str(blk))
        return "\n".join(parts)
    return str(content)


def _extract_tool_result(data) -> str:
    """on_tool_end의 event['data']에서 결과 텍스트만 추출 (항상 str 반환)."""
    if isinstance(data, dict) and "output" in data:
        out = data["output"]
        content = out.content if hasattr(out, "content") else out
    elif hasattr(data, "content"):
        content = data.content
    else:
        content = data
    return _stringify_content(content)


def _compact_result(text: str, limit: int = 140) -> str:
    """도구 결과를 한 줄로 압축 (compact)."""
    s = text.replace("\n", " ").strip()
    return s if len(s) <= limit else s[:limit] + "…"


# ─── 이미지 파일 업로드 ─────────────────────────────────────

_IMG_PATH_RE = re.compile(r"([^\s\"']+\.(?:png|jpg|jpeg|webp|gif))", re.IGNORECASE)


def _parse_image_path(result_text: str):
    """generate_image 결과 문자열에서 저장된 이미지 경로를 뽑습니다."""
    m = _IMG_PATH_RE.search(result_text or "")
    return m.group(1) if m else None


async def _upload_image(client, channel: str, thread_ts: str, path: str) -> bool:
    """생성된 이미지를 스레드에 파일로 첨부합니다. (files:write 스코프 필요)"""
    p = Path(path)
    if not p.exists():
        return False
    await client.files_upload_v2(
        channel=channel, thread_ts=thread_ts, file=str(p), title=p.name,
    )
    return True


class SlackStreamer:
    """astream_events 이벤트를 Slack 메시지로 점진 게시하는 헬퍼.

    하나의 활성 메시지 슬롯만 갱신한다. AI 토큰이 오면 그 메시지를 chat.update로
    토큰 단위 갱신(throttle 적용)하고, 도구 호출이 오면 진행 중 메시지를 확정한 뒤
    도구 메시지를 새로 게시한다. 다음 AI 토큰은 새 메시지를 시작해 시간순을 보존한다.

    출력 모드:
      - 'final'   : 도구 호출/결과 메시지를 Slack에 게시하지 않음.
      - 'compact' : 도구 호출/결과를 한 줄로 압축해서 게시 (기본).
      - 'verbose' : 도구 args/결과를 코드블록으로 풀 출력.
    """

    def __init__(self, web_client, channel: str, thread_ts: str, *,
                 mode: str = "compact",
                 min_update_interval: float = 1.0,
                 max_tool_result: int = 600):
        self.client = web_client
        self.channel = channel
        self.thread_ts = thread_ts
        self.mode = mode
        self.min_update_interval = min_update_interval
        self.max_tool_result = max_tool_result
        self._active_ts = None
        self._active_kind = None  # "placeholder" | "ai" | None
        self._active_text = ""
        self._last_update = 0.0

    async def _post(self, text: str) -> str:
        resp = await self.client.chat_postMessage(
            channel=self.channel, thread_ts=self.thread_ts,
            text=text or "_(빈 메시지)_",
        )
        return resp["ts"]

    async def _update(self, ts: str, text: str) -> None:
        try:
            await self.client.chat_update(
                channel=self.channel, ts=ts, text=text or "_(빈 메시지)_",
            )
        except Exception:
            # 레이트리밋/일시 오류: 다음 갱신이 따라잡으므로 조용히 무시.
            pass

    async def post_placeholder(self) -> None:
        self._active_ts = await self._post("_생각 중..._")
        self._active_kind = "placeholder"
        self._active_text = ""
        self._last_update = 0.0

    async def append_token(self, chunk: str) -> None:
        if self._active_kind == "ai":
            self._active_text += chunk
            now = time.monotonic()
            if now - self._last_update >= self.min_update_interval:
                await self._update(self._active_ts, self._active_text)
                self._last_update = now
            return

        # placeholder거나 활성 슬롯이 비어있음 → 새 AI 세그먼트 시작.
        self._active_text = chunk
        if self._active_ts is None:
            self._active_ts = await self._post(self._active_text)
        else:
            await self._update(self._active_ts, self._active_text)
        self._active_kind = "ai"
        self._last_update = time.monotonic()

    async def _finalize_active(self) -> None:
        """진행 중이던 메시지를 마지막 텍스트로 강제 동기화하고 슬롯을 비운다."""
        if self._active_ts and self._active_kind == "ai" and self._active_text:
            await self._update(self._active_ts, self._active_text)
        elif self._active_ts and self._active_kind == "placeholder":
            await self._update(self._active_ts, "_(응답 없음)_")
        self._active_ts = None
        self._active_kind = None
        self._active_text = ""

    def _format_tool_call(self, name: str, data) -> str:
        if self.mode == "verbose":
            return f"🔧 *Tool 호출*: `{name}`\n```\n{_format_args_full(data)}\n```"
        return f"🔧 `{name}` 호출 _({_compact_args(data)})_"

    def _format_tool_result(self, result_text: str) -> str:
        if self.mode == "verbose":
            truncated = (result_text if len(result_text) <= self.max_tool_result
                         else result_text[:self.max_tool_result] + "  …(생략)")
            return f"✅ *결과*\n```\n{truncated}\n```"
        return f"✅ _{_compact_result(result_text)}_"

    async def post_tool_call(self, name: str, data) -> None:
        if self.mode == "final":
            if self._active_kind == "ai":
                await self._finalize_active()
            return

        text = self._format_tool_call(name, data)
        if self._active_kind == "placeholder":
            await self._update(self._active_ts, text)
            self._active_ts = None
            self._active_kind = None
            self._active_text = ""
        else:
            await self._finalize_active()
            await self._post(text)

    async def post_tool_result(self, result_text: str) -> None:
        if self.mode == "final":
            return
        await self._post(self._format_tool_result(result_text))

    async def post_error(self, err_text: str) -> None:
        await self._finalize_active()
        await self._post(f"❌ 오류:\n```\n{err_text}\n```")

    async def close(self) -> None:
        """모든 이벤트 처리가 끝난 뒤 호출. 마지막 토큰까지 동기화 보장."""
        await self._finalize_active()


# ─── 봇 메인 ──────────────────────────────────────────────

async def main(mode: str):
    required = ["OPENAI_API_KEY", "SLACK_BOT_TOKEN", "SLACK_APP_TOKEN"]
    missing = [v for v in required if not os.getenv(v)]
    if missing:
        print(f"환경변수가 설정되지 않았습니다: {', '.join(missing)}")
        print("  .env 파일을 확인하세요.")
        sys.exit(1)

    stack = contextlib.AsyncExitStack()

    # 스레드 대화를 SQLite에 저장한다. 봇을 껐다 켜도 같은 스레드면 이어진다.
    checkpointer = await stack.enter_async_context(
        AsyncSqliteSaver.from_conn_string(CHECKPOINT_DB)
    )
    await checkpointer.setup()

    # stateful MCP 서버(예: Playwright)는 봇 lifetime 동안 세션을 열어 유지하고,
    # 그 도구를 build_agent에 extra_tools로 넘긴다. 세션은 stack에 넣어 종료 시 닫는다.
    # 설정은 mcp_servers.json의 stateful 항목에서 읽고, 연결에 실패한 서버는 건너뛴다.
    extra_tools = []
    stateful_servers = load_servers(stateful=True)
    if stateful_servers:
        sclient = MultiServerMCPClient(stateful_servers)
        for name in stateful_servers:
            try:
                session = await stack.enter_async_context(sclient.session(name))
                tools = await load_mcp_tools(session)
                extra_tools += tools
                print(f"{name} 도구 {len(tools)}개 로드 (stateful 세션)")
            except Exception as e:
                print(f"{name} 연결 실패({type(e).__name__}: {e}). 건너뜁니다.")

    print(f"에이전트 빌드 중... (출력 모드: {mode})")
    agent = await build_agent(
        system_prompt=SLACK_SYSTEM_PROMPT,
        checkpointer=checkpointer,
        extra_tools=extra_tools,
    )

    app = AsyncApp(token=os.environ["SLACK_BOT_TOKEN"])

    # 봇 자신의 user id. 멘션 판정과 루프 방지(봇 메시지 무시)에 쓴다.
    bot_user_id = (await app.client.auth_test())["user_id"]

    async def thread_is_active(thread_ts: str) -> bool:
        # 그 스레드에 이미 대화 기록(체크포인트)이 있으면 봇이 참여 중인 스레드다.
        cfg = {"configurable": {"thread_id": f"slack_{thread_ts}"}}
        return (await checkpointer.aget_tuple(cfg)) is not None

    @app.event("app_mention")
    async def absorb_app_mention(event):
        # 멘션은 message 이벤트로도 들어와 handle_message가 처리한다.
        # 여기서는 app_mention을 흡수만 해 중복 처리와 "Unhandled request" 경고를 막는다.
        pass

    @app.event("message")
    async def handle_message(event, client):
        # 루프 방지: 봇 자신/다른 봇/서브타입(편집, 파일첨부, 입퇴장) 메시지는 무시한다.
        if event.get("bot_id") or event.get("subtype") or event.get("user") == bot_user_id:
            return

        text = event.get("text", "") or ""
        channel = event["channel"]
        thread_ts = event.get("thread_ts", event["ts"])

        # 멘션이면 스레드를 연다. 멘션이 없어도 봇이 이미 참여 중인 스레드면 이어간다.
        # 둘 다 아니면(멘션 없는 일반 채널 대화) 무시한다.
        mentioned = f"<@{bot_user_id}>" in text
        if not (mentioned or await thread_is_active(thread_ts)):
            return

        body = text.replace(f"<@{bot_user_id}>", "").strip()
        if not body:
            await client.chat_postMessage(
                channel=channel, thread_ts=thread_ts, text="무엇을 도와드릴까요?"
            )
            return

        # 봇이 아는 채널/스레드 정보를 LLM 입력 앞에 붙여, Slack 도구 호출 시
        # 정확한 channel_id, thread_ts를 채우게 한다.
        context_block = (
            "[현재 Slack 컨텍스트]\n"
            f"채널 ID: {channel}\n"
            f"스레드 ts: {thread_ts}\n"
            "Slack 관련 도구를 호출할 때 이 값을 그대로 사용하세요.\n\n"
            "사용자 요청:\n"
            f"{body}"
        )

        streamer = SlackStreamer(client, channel, thread_ts, mode=mode)
        await streamer.post_placeholder()

        try:
            async for ev in agent.astream_events(
                {"messages": [{"role": "user", "content": context_block}]},
                version="v2",
                config={"configurable": {"thread_id": f"slack_{thread_ts}"}},
            ):
                kind = ev["event"]

                if kind == "on_chat_model_stream":
                    chunk = ev["data"]["chunk"].content
                    if chunk:
                        await streamer.append_token(chunk)

                elif kind == "on_tool_start":
                    await streamer.post_tool_call(ev["name"], ev["data"])

                elif kind == "on_tool_end":
                    result = _extract_tool_result(ev["data"])
                    await streamer.post_tool_result(result)
                    # generate_image 결과면 저장된 그림을 스레드에 파일로 첨부한다.
                    if ev["name"] == "generate_image":
                        path = _parse_image_path(result)
                        if path:
                            try:
                                await _upload_image(client, channel, thread_ts, path)
                            except Exception as e:
                                await streamer.post_error(
                                    f"이미지 업로드 실패: {type(e).__name__}: {e}"
                                )
        except Exception as e:
            await streamer.post_error(f"{type(e).__name__}: {e}")
        finally:
            await streamer.close()

    handler = AsyncSocketModeHandler(app, os.environ["SLACK_APP_TOKEN"])
    print("Slack App 시작 (Ctrl+C 로 종료)")
    try:
        await handler.start_async()
    finally:
        await stack.aclose()
        print("\n봇을 종료합니다...")


if __name__ == "__main__":
    mode = parse_args()
    try:
        asyncio.run(main(mode))
    except KeyboardInterrupt:
        pass


### 실행 방법

터미널을 3개 열어 MCP 서버와 봇을 실행합니다. image와 Playwright는 stdio라 봇이 자동으로 spawn하므로 따로 실행하지 않습니다.

```bash
# 터미널 1: Tavily 검색 MCP
python tavily_server.py

# 터미널 2: Slack MCP
python slack_mcp_server.py

# 터미널 3: 봇 (image, Playwright는 자동 spawn)
python slack_app.py             # = --mode compact (기본)
python slack_app.py --mode 1    # AI 텍스트만
python slack_app.py --mode 3    # 전체 풀 출력
```

출력 모드 (`--mode` / `-m`)
- 1 / final: AI 텍스트만
- 2 / compact (기본): AI 텍스트 풀, 도구 호출/결과는 한 줄 요약
- 3 / verbose: 도구 args와 결과를 코드블록으로 풀 출력


## 캡스톤: 나만의 기능을 얹어 Slack 봇에 배포하기

지금까지 만든 `build_agent`/`slack_app`은 네 개의 접합점으로 당신의 기능을 받아들입니다. 어디를 고치면 Slack 봇에 반영되는지 한곳에 정리합니다.

| 접합점 | 무엇을 | 어디에 | 반영 |
|---|---|---|---|
| 도구 (영구) | `@tool` 함수 | `tools.py`의 `DEFAULT_TOOLS` 리스트 | 봇 재시작 시 자동 |
| 도구 (즉석 테스트) | `@tool` 함수 | `build_agent(extra_tools=[...])` | 이 노트북에서 바로 |
| MCP 서버 | FastMCP 서버 파일 | `mcp_servers.json`의 `mcpServers` | 봇 재시작 시 자동 spawn |
| 미들웨어 | 05~08에서 만든 Middleware | `build_agent(middleware=[...])` | 컨텍스트/스킬/기억/승인 게이트 |

아래 스캐폴드의 `TODO`를 당신의 기능으로 채우고 실행해, 노트북에서 먼저 동작을 확인한 뒤 `tools.py`/`mcp_servers.json`에 옮겨 봇으로 배포하세요.


In [ ]:
# 캡스톤 스캐폴드: 아래 TODO를 당신의 기능으로 바꾸세요.
from langchain.tools import tool
from build_agent import build_agent

# --- 경로 A/B: 나만의 도구 -------------------------------------------------
@tool
def my_tool(query: str) -> str:
    """TODO: 도구 설명을 적으세요. LLM은 이 설명을 보고 도구를 고릅니다."""
    # TODO: 실제 로직으로 교체 (외부 API 호출, DB 조회 등)
    return f"내 도구가 받은 입력: {query}"

# 즉석 테스트: extra_tools로 주입해 이 노트북에서 바로 확인
agent = await build_agent(extra_tools=[my_tool])
result = await agent.ainvoke({
    'messages': [{'role': 'user', 'content': 'my_tool로 "hello"를 처리해줘'}]
})
print(result['messages'][-1].text)

# --- 봇으로 영구 배포하기 --------------------------------------------------
# 1) 위 my_tool을 tools.py로 옮기고 DEFAULT_TOOLS 리스트에 추가
# 2) (선택) 나만의 MCP 서버: mcp_demo_server.py를 복제해 my_server.py 작성 후
#    mcp_servers.json의 mcpServers에 한 줄 추가:
#        "my_server": {"transport": "stdio", "command": "python3", "args": ["my_server.py"]}
# 3) (선택) 05~08 미들웨어를 build_agent(middleware=[...])로 얹기
# 4) python slack_app.py 재시작 -> Slack에서 @봇 호출로 내 기능 확인
print("
[캡스톤 체크리스트] 내 기능이 Slack에 뜨면 완료입니다.")


---
## 6. 실서비스에서 추가로 신경 쓸 점

### 도구 설명도 프롬프트의 일부입니다

LLM은 도구를 고를 때 서버가 보낸 설명문(description)을 같이 읽습니다.  
신뢰할 수 없는 외부 MCP 서버(예: 외부 npm 패키지로 받은 Playwright MCP)를 붙이면, 그 설명문에 이상한 지시가 섞여 있을 수도 있다는 뜻입니다.  
사내에서 만든 서버만 붙이거나, 외부 서버는 별도로 검토하는 절차가 필요합니다.

### 토큰 관리

이번 노트북은 학습을 단순하게 하기 위해 `SLACK_BOT_TOKEN`을 환경변수에서 직접 읽어 썼습니다.  
실제 서비스에서는 보통 OAuth로 사용자별 토큰을 발급받고, 짧은 수명으로 자주 갱신하는 방식을 권장합니다.  
강의 범위를 넘어가는 주제이므로, 여기서는 production에서는 다르게 한다는 점만 기억해 주세요.


---
## 7. 정리

### 이번 노트북에서 다룬 것

- Elicitation: MCP 서버가 작업 도중 사용자에게 추가 정보를 되묻는 양방향 통신
  - 자동 응답 콜백(`on_elicitation`): 데모와 CI 환경에 유용
  - `input()` 기반 인터랙티브 콜백: 노트북에서 사용자에게 직접 묻기
- Slack MCP 서버: `slack_sdk` 기반으로 채널 조회 / 메시지 전송 / 리액션 / 스레드 답글 도구 제공
  - `SLACK_BOT_TOKEN` / `SLACK_CHANNEL_ID` 미설정 시 자동 dry-run 모드
  - `post_message`가 채널 미지정 시 Elicitation으로 사용자에게 채널 되묻기
- Stateless vs Stateful MCP: 호출 간 상태 공유 여부에 따른 클라이언트 패턴 구분
  - Stateless는 `client.get_tools()`로 ephemeral 세션을 사용합니다.
  - Stateful은 `async with client.session(...)`로 세션 수명을 명시적으로 관리합니다.
- 통합 에이전트 + Slack 봇 노출: 한 `MultiServerMCPClient`에 여러 MCP 서버를 묶고 (Playwright만 `client.session()`으로 stateful), `slack_bolt`(AsyncApp + Socket Mode)로 inbound 인터페이스를 제공합니다.  
  채널 컨텍스트는 매 멘션의 user message 앞에 prefix로 박아 Elicitation 없이 처리합니다.
- Playwright MCP (stdio + 별도 스크립트): 브라우저처럼 세션 단위 상태가 필요한 작업은 stdio 트랜스포트로 직접 붙이고, 노트북 셀 단위 한계를 피해 단일 파이썬 스크립트로 실행

### 더 보면 좋은 것

- MCP 공식 문서: <https://modelcontextprotocol.io>
- LangChain MCP Adapters: <https://github.com/langchain-ai/langchain-mcp-adapters>
